In [48]:
import joblib
import pandas as pd
from pathlib import Path

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder

In [49]:
# CONFIG

# กำหนด path ของข้อมูลและที่เก็บโมเดล
DATA_PATH = Path("data/cleaned_job_data_plus.csv")
MODEL_DIR = Path("model")
MODEL_DIR.mkdir(exist_ok=True)

MODEL_PATH = MODEL_DIR / "job_category_model.pkl"
LABEL_PATH = MODEL_DIR / "label_encoder.pkl"

In [50]:
# โหลดข้อมูล
if not DATA_PATH.exists():
    raise FileNotFoundError(f"ไม่พบไฟล์: {DATA_PATH}")

df = pd.read_csv(DATA_PATH)

print("โหลดข้อมูลสำเร็จ")
print("Shape:", df.shape)
df.head()

โหลดข้อมูลสำเร็จ
Shape: (1095, 11)


,Category,Workplace,Location,Department,Type,final_features,Skills,Keywords,Description,Experience,final_features_plus
0,business analyst,remote,united kingdom,operations,full time,business analyst remote united kingdom operati...,excel sql reporting dashboard business analysi...,business analyst operations full time united k...,business analyst role in operations department...,mid,business analyst remote united kingdom operati...
1,business analyst,remote,makati metro manila philippines,aux hq,full time,business analyst remote makati metro manila ph...,figma wireframe prototype user research visual...,business analyst aux hq full time makati metro...,business analyst role in aux hq department thi...,mid,business analyst remote makati metro manila ph...
2,business analyst,on site,al dajeej al farwaniyah governorate kuwait,pwc technologies,full time,business analyst on site al dajeej al farwaniy...,excel sql reporting dashboard business analysi...,business analyst pwc technologies full time al...,business analyst role in pwc technologies depa...,mid,business analyst on site al dajeej al farwaniy...
3,business analyst,on site,london england united kingdom,consultants advisory,full time,business analyst on site london england united...,excel sql reporting dashboard business analysi...,business analyst consultants advisory full tim...,business analyst role in consultants advisory ...,mid,business analyst on site london england united...
4,business analyst,remote,united kingdom,operations,full time,business analyst remote united kingdom operati...,excel sql reporting dashboard business analysi...,business analyst operations full time united k...,business analyst role in operations department...,mid,business analyst remote united kingdom operati...


In [51]:
# ต้องมีทั้งข้อมูลจริงและข้อมูล plus
required_cols = ["Category", "final_features", "final_features_plus"]

for col in required_cols:
    if col not in df.columns:
        raise ValueError(f"ไม่พบคอลัมน์สำคัญ: {col}")

df[required_cols].head()

,Category,final_features,final_features_plus
0,business analyst,business analyst remote united kingdom operati...,business analyst remote united kingdom operati...
1,business analyst,business analyst remote makati metro manila ph...,business analyst remote makati metro manila ph...
2,business analyst,business analyst on site al dajeej al farwaniy...,business analyst on site al dajeej al farwaniy...
3,business analyst,business analyst on site london england united...,business analyst on site london england united...
4,business analyst,business analyst remote united kingdom operati...,business analyst remote united kingdom operati...


In [52]:
# clean ข้อมูลก่อนใช้
df["Category"] = df["Category"].fillna("unknown").astype(str).str.strip()
df["final_features"] = df["final_features"].fillna("unknown").astype(str).str.strip()
df["final_features_plus"] = df["final_features_plus"].fillna("unknown").astype(str).str.strip()

df = df[
    (df["Category"] != "") &
    (df["Category"] != "unknown") &
    (df["final_features"] != "") &
    (df["final_features"] != "unknown") &
    (df["final_features_plus"] != "") &
    (df["final_features_plus"] != "unknown")
].copy()

df = df.drop_duplicates(subset=["Category", "final_features", "final_features_plus"]).reset_index(drop=True)

print("Shape หลัง clean:", df.shape)
print(df["Category"].value_counts())

Shape หลัง clean: (672, 11)
Category
data scientist        135
cloud                 120
hr                    112
business analyst      108
software developer    106
ui ux                  91
Name: count, dtype: int64


In [53]:
# แปลง label เป็นตัวเลข
label_encoder = LabelEncoder()
df["label"] = label_encoder.fit_transform(df["Category"])

print("Classes:", list(label_encoder.classes_))

Classes: ['business analyst', 'cloud', 'data scientist', 'hr', 'software developer', 'ui ux']


In [54]:
# split โดยใช้ index เดียวกันก่อน
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df["label"],
    random_state=42
)

print("Train size:", len(train_df))
print("Test size:", len(test_df))

Train size: 537
Test size: 135


In [55]:
# ฝั่ง train ใช้ข้อมูล plus
X_train = train_df["final_features_plus"]

# ฝั่ง test ใช้ข้อมูลจริง
X_test = test_df["final_features"]

y_train = train_df["label"]
y_test = test_df["label"]

print("ตัวอย่าง train text:")
print(X_train.head(3).tolist())

print("\nตัวอย่าง test text:")
print(X_test.head(3).tolist())

ตัวอย่าง train text:
['ui ux remote riyadh riyadh province saudi arabia product development full time figma wireframe prototype user research visual design ui ux product development full time riyadh riyadh province saudi arabia remote ui ux role in product development department this is a full time position based in riyadh riyadh province saudi arabia with workplace style remote preferred skills include figma wireframe prototype user research visual design the role requires communication teamwork and problem solving mid', 'data scientist on site athens attica greece gek terna ict full time python sql pandas numpy machine learning statistics data visualization data scientist gek terna ict full time athens attica greece on site data scientist role in gek terna ict department this is a full time position based in athens attica greece with workplace style on site preferred skills include python sql pandas numpy machine learning statistics data visualization the role requires communication 

In [56]:
# สร้าง pipeline
model_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        max_features=10000,
        ngram_range=(1, 3),
        min_df=1,
        max_df=0.95,
        sublinear_tf=True,
        stop_words="english"
    )),
    ("clf", LinearSVC(
        class_weight="balanced",
        random_state=42
    ))
])

In [57]:
# train model
model_pipeline.fit(X_train, y_train)
print("Train Success")

Train Success


In [58]:
# evaluate model
y_pred = model_pipeline.predict(X_test)

acc = accuracy_score(y_test, y_pred)
print(f"Accuracy (train=plus, test=real): {acc:.4f}")

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))

Accuracy (train=plus, test=real): 0.9556

Classification Report:

                    precision    recall  f1-score   support

  business analyst       0.96      1.00      0.98        22
             cloud       1.00      0.75      0.86        24
    data scientist       0.93      1.00      0.96        27
                hr       0.96      1.00      0.98        23
software developer       0.91      1.00      0.95        21
             ui ux       1.00      1.00      1.00        18

          accuracy                           0.96       135
         macro avg       0.96      0.96      0.96       135
      weighted avg       0.96      0.96      0.95       135



In [59]:
# confusion matrix
cm = confusion_matrix(y_test, y_pred)
cm_df = pd.DataFrame(cm, index=label_encoder.classes_, columns=label_encoder.classes_)

print("Confusion Matrix:")
cm_df

Confusion Matrix:


,business analyst,cloud,data scientist,hr,software developer,ui ux
business analyst,22,0,0,0,0,0
cloud,1,18,2,1,2,0
data scientist,0,0,27,0,0,0
hr,0,0,0,23,0,0
software developer,0,0,0,0,21,0
ui ux,0,0,0,0,0,18


In [60]:
# cross validation แบบ train plus / test real ทำเอง
from sklearn.model_selection import StratifiedKFold
import numpy as np

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = []

for fold, (train_idx, test_idx) in enumerate(skf.split(df, df["label"]), start=1):
    train_fold = df.iloc[train_idx]
    test_fold = df.iloc[test_idx]

    X_train_fold = train_fold["final_features_plus"]
    X_test_fold = test_fold["final_features"]
    y_train_fold = train_fold["label"]
    y_test_fold = test_fold["label"]

    fold_pipeline = Pipeline([
        ("tfidf", TfidfVectorizer(
            max_features=10000,
            ngram_range=(1, 3),
            min_df=1,
            max_df=0.95,
            sublinear_tf=True,
            stop_words="english"
        )),
        ("clf", LinearSVC(
            class_weight="balanced",
            random_state=42
        ))
    ])

    fold_pipeline.fit(X_train_fold, y_train_fold)
    y_pred_fold = fold_pipeline.predict(X_test_fold)

    fold_acc = accuracy_score(y_test_fold, y_pred_fold)
    cv_scores.append(fold_acc)

    print(f"Fold {fold}: {fold_acc:.4f}")

print("\nCV scores:", cv_scores)
print("Mean CV accuracy:", np.mean(cv_scores))

Fold 1: 0.9407
Fold 2: 0.9778
Fold 3: 0.9701
Fold 4: 0.9776
Fold 5: 0.9701

CV scores: [0.9407407407407408, 0.9777777777777777, 0.9701492537313433, 0.9776119402985075, 0.9701492537313433]
Mean CV accuracy: 0.9672857932559425


In [61]:
# save model
joblib.dump(model_pipeline, MODEL_PATH)
joblib.dump(label_encoder, LABEL_PATH)

print(f"บันทึก model สำเร็จ: {MODEL_PATH}")
print(f"บันทึก label encoder สำเร็จ: {LABEL_PATH}")

บันทึก model สำเร็จ: model\job_category_model.pkl
บันทึก label encoder สำเร็จ: model\label_encoder.pkl


In [62]:
# ทดลอง predict ข้อมูลใหม่แบบ real text
sample_texts = [
    "python sql machine learning data analysis",
    "aws docker kubernetes devops cloud",
    "figma ux ui wireframe prototype",
    "recruitment onboarding payroll hr"
]

sample_preds = model_pipeline.predict(sample_texts)
sample_labels = label_encoder.inverse_transform(sample_preds)

for text, label in zip(sample_texts, sample_labels):
    print("TEXT:", text)
    print("PREDICT:", label)
    print("-" * 50)

TEXT: python sql machine learning data analysis
PREDICT: data scientist
--------------------------------------------------
TEXT: aws docker kubernetes devops cloud
PREDICT: cloud
--------------------------------------------------
TEXT: figma ux ui wireframe prototype
PREDICT: ui ux
--------------------------------------------------
TEXT: recruitment onboarding payroll hr
PREDICT: hr
--------------------------------------------------


In [63]:
from pathlib import Path

MODEL_PATH = Path("model/job_category_model.pkl")
LABEL_PATH = Path("model/label_encoder.pkl")

print("MODEL absolute path:", MODEL_PATH.resolve())
print("LABEL absolute path:", LABEL_PATH.resolve())
print("MODEL size:", MODEL_PATH.stat().st_size)
print("LABEL size:", LABEL_PATH.stat().st_size)

MODEL absolute path: C:\Users\note3\ProjectMLops\model\job_category_model.pkl
LABEL absolute path: C:\Users\note3\ProjectMLops\model\label_encoder.pkl
MODEL size: 857784
LABEL size: 551
